# 01 - MP4 to Cropped Labelled Frames

This notebook reads Copernicus MP4 + timeline TXT pairs from `raw_to_be_processed/`, archives the raw files into `data/raw/<label>/<latitude>_<longitude>/`, extracts one frame per timeline date, crops 5% from every border, and saves processed PNG frames into `data/processed/<label>/<latitude>_<longitude>/`.

Input naming:

```text
raw_to_be_processed/
  3.5165528_101.9364861_high.mp4
  3.5165528_101.9364861.txt
  # also accepted: 3.5165528_101.9364861_high.txt
```

Timeline rule: line 1 maps to timestamp `0s`, line 2 maps to `1s`, and so on.

In [15]:
from pathlib import Path
import importlib.util
import json
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

INBOX_DIR = PROJECT_ROOT / "raw_to_be_processed"
DATA_DIR = PROJECT_ROOT / "data"
CROP_PERCENT = 5.0
FORCE_REPROCESS = False

print("Project root:", PROJECT_ROOT)
print("Inbox:", INBOX_DIR)
print("Data dir:", DATA_DIR)
print("OpenCV available:", importlib.util.find_spec("cv2") is not None)

Project root: c:\Users\Aki\Downloads\AML - Final Project\code
Inbox: c:\Users\Aki\Downloads\AML - Final Project\code\raw_to_be_processed
Data dir: c:\Users\Aki\Downloads\AML - Final Project\code\data
OpenCV available: True


## Dependency Check

This pipeline needs `opencv-python` to decode MP4 frames. If OpenCV is not available, install it first:

```powershell
pip install -r requirements.txt
```

or:

```powershell
pip install opencv-python
```

In [16]:
if importlib.util.find_spec("cv2") is None:
    raise ImportError(
        "opencv-python is not installed. Run `pip install -r requirements.txt` "
        "from the project root, then restart this notebook kernel."
    )

## Preview Incoming Samples

In [17]:
from src.preprocessing.process_raw_videos import discover_samples, read_timeline

samples = discover_samples(INBOX_DIR)
preview_rows = []
for sample in samples:
    dates = read_timeline(sample.timeline_path)
    preview_rows.append({
        "sample_id": sample.sample_id,
        "label": sample.label,
        "video": sample.video_path.name,
        "timeline": sample.timeline_path.name,
        "frame_count": len(dates),
        "start_date": dates[0],
        "end_date": dates[-1],
    })

preview_rows

[{'sample_id': '3.3908_102.01211',
  'label': 'high',
  'video': '3.3908_102.01211_high.mp4',
  'timeline': '3.3908_102.01211_high.txt',
  'frame_count': 18,
  'start_date': '2016-06-08',
  'end_date': '2025-10-04'},
 {'sample_id': '3.40614_102.05138',
  'label': 'high',
  'video': '3.40614_102.05138_high.mp4',
  'timeline': '3.40614_102.05138_high.txt',
  'frame_count': 18,
  'start_date': '2016-06-08',
  'end_date': '2025-10-04'},
 {'sample_id': '3.42216_102.01648',
  'label': 'moderate',
  'video': '3.42216_102.01648_moderate.mp4',
  'timeline': '3.42216_102.01648_moderate.txt',
  'frame_count': 18,
  'start_date': '2016-06-08',
  'end_date': '2025-07-11'},
 {'sample_id': '3.44071_101.66564',
  'label': 'low',
  'video': '3.44071_101.66564_low.mp4',
  'timeline': '3.44071_101.66564_low.txt',
  'frame_count': 47,
  'start_date': '2016-01-30',
  'end_date': '2026-06-06'},
 {'sample_id': '3.4521_102.0134',
  'label': 'high',
  'video': '3.4521_102.0134_high.mp4',
  'timeline': '3.4521_

## Process MP4 Videos

Expected output for each sample:

```text
data/raw/<label>/<latitude>_<longitude>/
  original_video.mp4
  timeline.txt

data/processed/<label>/<latitude>_<longitude>/
  frame_000__YYYY-MM-DD.png
  frame_001__YYYY-MM-DD.png
  frame_metadata.csv
  processing_metadata.json
```

In [18]:
import importlib
import src.preprocessing.process_raw_videos as raw_video_processing

raw_video_processing = importlib.reload(raw_video_processing)
process_inbox = raw_video_processing.process_inbox
write_sample_index = raw_video_processing.write_sample_index

results = process_inbox(
    inbox_dir=INBOX_DIR,
    data_dir=DATA_DIR,
    crop_percent=CROP_PERCENT,
    force=FORCE_REPROCESS,
)
sample_index_path = write_sample_index(DATA_DIR)

print(json.dumps(results, indent=2))
print("Sample index:", sample_index_path)


[
  {
    "latitude": 3.3908,
    "longitude": 102.01211,
    "label": "high",
    "sample_id": "3.3908_102.01211_high",
    "coordinate_id": "3.3908_102.01211",
    "raw_dir": "c:\\Users\\Aki\\Downloads\\AML - Final Project\\code\\data\\raw\\high\\3.3908_102.01211",
    "raw_video": "c:\\Users\\Aki\\Downloads\\AML - Final Project\\code\\data\\raw\\high\\3.3908_102.01211\\3.3908_102.01211_high.mp4",
    "raw_timeline": "c:\\Users\\Aki\\Downloads\\AML - Final Project\\code\\data\\raw\\high\\3.3908_102.01211\\3.3908_102.01211.txt",
    "processed_dir": "c:\\Users\\Aki\\Downloads\\AML - Final Project\\code\\data\\processed\\high\\3.3908_102.01211",
    "frame_count": 18,
    "crop_percent": 5.0,
    "status": "processed",
    "processed_at": "2026-06-22T08:37:53.869254+00:00",
    "gee_observations_csv": "c:\\Users\\Aki\\Downloads\\AML - Final Project\\code\\data\\processed\\high\\3.3908_102.01211\\gee_observations.csv",
    "gee_features_csv": "c:\\Users\\Aki\\Downloads\\AML - Final Proj

## Verify Sample Index

`data/processed/sample_index.csv` is the central coordinate listing used by `02_extract_gee_features.ipynb`.


In [19]:
import csv

with sample_index_path.open(newline="", encoding="utf-8") as handle:
    sample_index_rows = list(csv.DictReader(handle))

print("Indexed samples:", len(sample_index_rows))
print("Labels:", sorted({row["label"] for row in sample_index_rows}))
sample_index_rows[:3]


Indexed samples: 10
Labels: ['high', 'low', 'moderate']


[{'sample_id': '3.3908_102.01211_high',
  'label': 'high',
  'latitude': '3.3908',
  'longitude': '102.01211',
  'processed_dir': 'c:\\Users\\Aki\\Downloads\\AML - Final Project\\code\\data\\processed\\high\\3.3908_102.01211',
  'frame_count': '18',
  'start_date': '2016-06-08',
  'end_date': '2025-10-04',
  'frame_metadata_csv': 'c:\\Users\\Aki\\Downloads\\AML - Final Project\\code\\data\\processed\\high\\3.3908_102.01211\\frame_metadata.csv',
  'processing_metadata_json': 'c:\\Users\\Aki\\Downloads\\AML - Final Project\\code\\data\\processed\\high\\3.3908_102.01211\\processing_metadata.json',
  'gee_observations_csv': 'c:\\Users\\Aki\\Downloads\\AML - Final Project\\code\\data\\processed\\high\\3.3908_102.01211\\gee_observations.csv',
  'gee_features_csv': 'c:\\Users\\Aki\\Downloads\\AML - Final Project\\code\\data\\processed\\high\\3.3908_102.01211\\gee_features.csv',
  'gee_targets_csv': 'c:\\Users\\Aki\\Downloads\\AML - Final Project\\code\\data\\processed\\high\\3.3908_102.01211\

## Verify Processed Output

In [20]:
for result in results:
    processed_dir = Path(result["processed_dir"])
    frame_paths = sorted(processed_dir.glob("frame_*.png"))
    print(processed_dir)
    print("PNG frame count:", len(frame_paths))
    print("Frame metadata exists:", (processed_dir / "frame_metadata.csv").exists())
    print("Processing metadata exists:", (processed_dir / "processing_metadata.json").exists())
    print("First frames:", [path.name for path in frame_paths[:3]])
    print("Last frames:", [path.name for path in frame_paths[-3:]])

c:\Users\Aki\Downloads\AML - Final Project\code\data\processed\high\3.3908_102.01211
PNG frame count: 18
Frame metadata exists: True
Processing metadata exists: True
First frames: ['frame_000__2016-06-08.png', 'frame_001__2017-01-14.png', 'frame_002__2018-08-12.png']
Last frames: ['frame_015__2024-03-23.png', 'frame_016__2025-07-11.png', 'frame_017__2025-10-04.png']
c:\Users\Aki\Downloads\AML - Final Project\code\data\processed\high\3.40614_102.05138
PNG frame count: 18
Frame metadata exists: True
Processing metadata exists: True
First frames: ['frame_000__2016-06-08.png', 'frame_001__2017-01-14.png', 'frame_002__2018-08-12.png']
Last frames: ['frame_015__2024-03-23.png', 'frame_016__2025-07-11.png', 'frame_017__2025-10-04.png']
c:\Users\Aki\Downloads\AML - Final Project\code\data\processed\moderate\3.42216_102.01648
PNG frame count: 18
Frame metadata exists: True
Processing metadata exists: True
First frames: ['frame_000__2016-06-08.png', 'frame_001__2017-01-14.png', 'frame_002__2018-